# VitalDB — ICU Stay Duration Prediction (Ordinal)

Predicts ICU stay duration as a 3-class ordinal outcome: 0 = No ICU, 1 = Short Stay (1–3 days), 2 = Prolonged Stay (4+ days). The 1-day cutoff was chosen because 68% of ICU admissions in this cohort lasted exactly one day, consistent with routine postoperative observation rather than significant deterioration.

**Methodology:** identical to the ICU admission notebook — 80/20 stratified holdout first, 5-fold CV with SMOTE inside folds only, bootstrap CI on holdout, six-model comparison structure (Signal only → Clinical only → Vitals only → Signal+Clinical → Clinical+Vitals → Full). AUC-ROC and AUC-PR are macro-averaged one-vs-rest across the three classes.

Requires `clinical_data.csv`, `vitaldb_ppg_features.csv`, `vitaldb_ecg_features.csv`, `vitaldb_hr_spo2.csv` (see Notebook 1 for extraction).

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import shap
import matplotlib as mpl
import matplotlib.pyplot as plt
import json, warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Data and Define Ordinal Outcome

In [ ]:
df        = pd.read_csv('clinical_data.csv')
vitals_df = pd.read_csv('vitaldb_hr_spo2.csv')
ecg_df    = pd.read_csv('vitaldb_ecg_features.csv')
ppg_df    = pd.read_csv('vitaldb_ppg_features.csv')

data = df.copy()
data['ecg_abnormal'] = (data['preop_ecg'] != 'Normal Sinus Rhythm').astype(int)
for col in ['sex', 'department', 'optype']:
    if col in data.columns:
        data[col] = LabelEncoder().fit_transform(data[col].astype(str))

def icu_ordinal(days):
    if days <= 0:   return 0
    elif days <= 3: return 1
    else:           return 2

data['icu_ordinal'] = data['icu_days'].apply(icu_ordinal)
print("Ordinal distribution (0=No ICU, 1=Short 1-3d, 2=Prolonged 4+d):")
print(data['icu_ordinal'].value_counts().sort_index())

FEATURES_CLINICAL = [
    'age', 'sex', 'bmi', 'asa', 'emop',
    'preop_htn', 'preop_dm', 'ecg_abnormal',
    'preop_hb', 'preop_plt', 'preop_pt', 'preop_aptt',
    'preop_na', 'preop_k', 'preop_gluc', 'preop_alb',
    'preop_cr', 'preop_bun', 'preop_ast', 'preop_alt'
]
FEATURES_VITALS = ['hr_mean', 'hr_min', 'hr_max', 'hr_std', 'spo2_mean', 'spo2_min']
FEATURES_PPG = [
    'ppg_mean', 'ppg_median', 'ppg_std', 'ppg_iqr', 'ppg_range',
    'ppg_skew', 'ppg_kurtosis', 'ppg_energy', 'ppg_mad', 'ppg_min', 'ppg_max'
]
FEATURES_ECG = [
    'ecg_mean', 'ecg_median', 'ecg_std', 'ecg_iqr', 'ecg_range',
    'ecg_skew', 'ecg_kurtosis', 'ecg_energy', 'ecg_mad', 'ecg_min', 'ecg_max'
]

## 2. Shared Pipeline Functions (Ordinal, Macro AUC-ROC + AUC-PR)

In [ ]:
def prep_X(df, cols):
    X = df[cols].copy().apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = X[col].fillna(X[col].median())
    return X


def macro_avg_precision(y_true, y_proba):
    """One-vs-rest average precision, macro averaged across classes."""
    n_classes = y_proba.shape[1]
    aps = []
    for c in range(n_classes):
        y_bin = (y_true == c).astype(int)
        aps.append(average_precision_score(y_bin, y_proba[:, c]))
    return np.mean(aps)


def run_cv_ordinal(X, y, label):
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    smote = SMOTE(random_state=RANDOM_STATE)
    oof_preds = np.zeros((len(y), 3))
    fold_aucs, fold_prs = [], []
    for tr_idx, val_idx in cv.split(X, y):
        X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
        m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          objective='multi:softprob', num_class=3,
                          random_state=RANDOM_STATE, verbosity=0)
        m.fit(X_tr_sm, y_tr_sm)
        proba = m.predict_proba(X_val)
        oof_preds[val_idx] = proba
        fold_aucs.append(roc_auc_score(y_val, proba, multi_class='ovr', average='macro'))
        fold_prs.append(macro_avg_precision(y_val.values, proba))
    cv_auc = roc_auc_score(y, oof_preds, multi_class='ovr', average='macro')
    cv_pr  = macro_avg_precision(y.values, oof_preds)
    print(f"{label}: CV Macro AUC-ROC {cv_auc:.4f} \u00b1 {np.std(fold_aucs):.4f} | CV Macro AUC-PR {cv_pr:.4f} \u00b1 {np.std(fold_prs):.4f}")
    X_sm, y_sm = smote.fit_resample(X, y)
    final = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          objective='multi:softprob', num_class=3,
                          random_state=RANDOM_STATE, verbosity=0)
    final.fit(X_sm, y_sm)
    return final, cv_auc, cv_pr


def bootstrap_ci_metric(y_true, y_proba, metric_fn, n=2000):
    scores = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 3:
            continue
        scores.append(metric_fn(y_true[idx], y_proba[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


def bootstrap_delta_macro(y_true, proba_a, proba_b, n=2000):
    deltas = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 3:
            continue
        a = roc_auc_score(y_true[idx], proba_a[idx], multi_class='ovr', average='macro')
        b = roc_auc_score(y_true[idx], proba_b[idx], multi_class='ovr', average='macro')
        deltas.append(b - a)
    return np.mean(deltas), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)


def eval_holdout_ordinal(model, X_test, y_test, label):
    proba  = model.predict_proba(X_test)
    y_arr  = np.array(y_test)
    auc_mean, auc_lo, auc_hi = bootstrap_ci_metric(y_arr, proba, lambda y, p: roc_auc_score(y, p, multi_class='ovr', average='macro'))
    pr_mean,  pr_lo,  pr_hi  = bootstrap_ci_metric(y_arr, proba, macro_avg_precision)
    print(f"{label} \u2014 HOLDOUT: Macro AUC-ROC {auc_mean:.4f} ({auc_lo:.4f}-{auc_hi:.4f}) | Macro AUC-PR {pr_mean:.4f} ({pr_lo:.4f}-{pr_hi:.4f})")
    return proba, {'macro_auc': auc_mean, 'auc_ci': (auc_lo, auc_hi), 'macro_pr': pr_mean, 'pr_ci': (pr_lo, pr_hi)}

print("Pipeline functions defined.")

## 3. ECG Analysis — Six-Model Comparison

In [ ]:
merged_ecg = data.merge(vitals_df, on='caseid', how='inner').merge(ecg_df, on='caseid', how='inner')
print(f"ECG cohort: {len(merged_ecg)}")
print(merged_ecg['icu_ordinal'].value_counts().sort_index())

train_ecg, test_ecg = train_test_split(merged_ecg, test_size=0.2, stratify=merged_ecg['icu_ordinal'], random_state=RANDOM_STATE)
train_ecg = train_ecg.reset_index(drop=True)
test_ecg  = test_ecg.reset_index(drop=True)

y_train_ecg = train_ecg['icu_ordinal'].astype(int)
y_test_ecg  = test_ecg['icu_ordinal'].astype(int)

ecg_models, ecg_results = {}, {}

for name, feats in [
    ('ecg_only', FEATURES_ECG),
    ('clinical_only', FEATURES_CLINICAL),
    ('vitals_only', FEATURES_VITALS),
    ('ecg_clinical', FEATURES_ECG + FEATURES_CLINICAL),
    ('clinical_vitals', FEATURES_CLINICAL + FEATURES_VITALS),
    ('full', FEATURES_ECG + FEATURES_CLINICAL + FEATURES_VITALS),
]:
    X_tr = prep_X(train_ecg, feats)
    X_te = prep_X(test_ecg, feats)
    model, cv_auc, cv_pr = run_cv_ordinal(X_tr, y_train_ecg, name)
    proba, res = eval_holdout_ordinal(model, X_te, y_test_ecg, name)
    res['cv_auc'] = cv_auc
    res['cv_pr']  = cv_pr
    ecg_models[name]  = model
    ecg_results[name] = {'proba': proba, 'metrics': res}

## 4. PPG Analysis — Six-Model Comparison

In [ ]:
merged_ppg = data.merge(vitals_df, on='caseid', how='inner').merge(ppg_df, on='caseid', how='inner')
print(f"PPG cohort: {len(merged_ppg)}")
print(merged_ppg['icu_ordinal'].value_counts().sort_index())

train_ppg, test_ppg = train_test_split(merged_ppg, test_size=0.2, stratify=merged_ppg['icu_ordinal'], random_state=RANDOM_STATE)
train_ppg = train_ppg.reset_index(drop=True)
test_ppg  = test_ppg.reset_index(drop=True)

y_train_ppg = train_ppg['icu_ordinal'].astype(int)
y_test_ppg  = test_ppg['icu_ordinal'].astype(int)

ppg_models, ppg_results = {}, {}

for name, feats in [
    ('ppg_only', FEATURES_PPG),
    ('clinical_only', FEATURES_CLINICAL),
    ('vitals_only', FEATURES_VITALS),
    ('ppg_clinical', FEATURES_PPG + FEATURES_CLINICAL),
    ('clinical_vitals', FEATURES_CLINICAL + FEATURES_VITALS),
    ('full', FEATURES_PPG + FEATURES_CLINICAL + FEATURES_VITALS),
]:
    X_tr = prep_X(train_ppg, feats)
    X_te = prep_X(test_ppg, feats)
    model, cv_auc, cv_pr = run_cv_ordinal(X_tr, y_train_ppg, name)
    proba, res = eval_holdout_ordinal(model, X_te, y_test_ppg, name)
    res['cv_auc'] = cv_auc
    res['cv_pr']  = cv_pr
    ppg_models[name]  = model
    ppg_results[name] = {'proba': proba, 'metrics': res}

## 5. Incremental Comparisons (Bootstrap Delta Macro AUC)

In [ ]:
def print_deltas_ordinal(results, y_test, signal_name):
    y_arr = np.array(y_test)
    d1 = bootstrap_delta_macro(y_arr, results['clinical_only']['proba'], results[f'{signal_name}_clinical']['proba'])
    d2 = bootstrap_delta_macro(y_arr, results[f'{signal_name}_only']['proba'], results[f'{signal_name}_clinical']['proba'])
    d3 = bootstrap_delta_macro(y_arr, results['clinical_vitals']['proba'], results[f'{signal_name}_clinical']['proba'])
    d4 = bootstrap_delta_macro(y_arr, results[f'{signal_name}_clinical']['proba'], results['full']['proba'])
    d5 = bootstrap_delta_macro(y_arr, results['clinical_vitals']['proba'], results['full']['proba'])
    print(f"\n{signal_name.upper()} deltas:")
    print(f"  Signal+Clin over Clin alone:        {d1[0]:+.4f} ({d1[1]:+.4f} to {d1[2]:+.4f})")
    print(f"  Signal+Clin over Signal alone:       {d2[0]:+.4f} ({d2[1]:+.4f} to {d2[2]:+.4f})")
    print(f"  Signal+Clin vs Vitals+Clin:          {d3[0]:+.4f} ({d3[1]:+.4f} to {d3[2]:+.4f})")
    print(f"  Full over Signal+Clin (vitals' val): {d4[0]:+.4f} ({d4[1]:+.4f} to {d4[2]:+.4f})")
    print(f"  Full over Clin+Vitals (signal's val): {d5[0]:+.4f} ({d5[1]:+.4f} to {d5[2]:+.4f})")

print_deltas_ordinal(ecg_results, y_test_ecg, 'ecg')
print_deltas_ordinal(ppg_results, y_test_ppg, 'ppg')

## 6. Save Results

In [ ]:
def serialize(results):
    return {name: r['metrics'] for name, r in results.items()}

all_results = {'ecg': serialize(ecg_results), 'ppg': serialize(ppg_results)}
with open('vitaldb_icu_stay_duration_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print("Saved to vitaldb_icu_stay_duration_results.json")